# 05 · National Aggregation & PHAC Validation

Aggregates weekly provincial model predictions (LSTM and ARIMA) to national
annual rates, then compares against PHAC-reported annual incidence as a
directional sanity check.

**Scope**: Alberta (CA-AB) + Ontario (CA-ON) only — ~50 % of Canada's population.
Population-weighted averaging is used to compute a representative national rate.
Comparisons are directional (trend / magnitude) rather than a full national replication.

**Test period**: 2004–2021 (matches LSTM and ARIMA test splits)

**Inputs**
- `reports/tables/lstm_weekly_predictions.csv` — from notebook 03B (cell 11.6)
- `reports/tables/arima_test_forecasts.csv` — from notebook 03A
- `data/processed/phac-clean.csv` — PHAC annual national incidence
- `data/processed/final_modeling_dataset.csv` — provincial population data

**Outputs**
- `reports/tables/national_annual_predictions.csv` — annual national rates by model and disease
- `reports/tables/phac_validation_comparison.csv` — PHAC vs model comparison with error metrics
- `reports/figures/phac_validation_*.png` — time series and error visualisations

---
## 1 · Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore")

REPO_DIR    = "/Users/deansimmer/git/AAI-590-capstone-canadian-health"
TABLES_DIR  = os.path.join(REPO_DIR, "reports", "tables")
FIGURES_DIR = os.path.join(REPO_DIR, "reports", "figures")

os.makedirs(TABLES_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# PHAC uses 'pertussis'; our models use 'whooping-cough'
DISEASE_MAP = {"whooping-cough": "pertussis"}
DISEASES    = ["influenza", "measles", "whooping-cough"]
PROVINCES   = ["CA-AB", "CA-ON"]

print("Paths configured.")

---
## 2 · Load model predictions

In [ ]:
# ── LSTM weekly predictions (exported by notebook 03B, cell 11.6) ─────────────
lstm_preds = pd.read_csv(
    os.path.join(TABLES_DIR, "lstm_weekly_predictions.csv"),
    parse_dates=["forecast_date"],
)
lstm_preds["year"] = lstm_preds["forecast_date"].dt.year
print(f"LSTM predictions : {len(lstm_preds):,} rows")
print(f"  Date range     : {lstm_preds['forecast_date'].min().date()} → {lstm_preds['forecast_date'].max().date()}")
print(f"  Diseases       : {sorted(lstm_preds['disease'].unique())}")
print(f"  Provinces      : {sorted(lstm_preds['province'].unique())}")
lstm_preds.head(3)

In [ ]:
# ── ARIMA weekly forecasts ────────────────────────────────────────────────────
arima_preds = pd.read_csv(
    os.path.join(TABLES_DIR, "arima_test_forecasts.csv"),
    parse_dates=["date"],
).rename(columns={"date": "forecast_date", "pred_cases_per_100k": "pred_cases_per_100k",
                   "actual_cases_per_100k": "actual_cases_per_100k"})
arima_preds["year"] = arima_preds["forecast_date"].dt.year
print(f"ARIMA predictions: {len(arima_preds):,} rows")
print(f"  Date range     : {arima_preds['forecast_date'].min().date()} → {arima_preds['forecast_date'].max().date()}")
print(f"  Diseases       : {sorted(arima_preds['disease'].unique())}")
print(f"  Provinces      : {sorted(arima_preds['province'].unique())}")
arima_preds.head(3)

---
## 3 · Provincial population by year

In [ ]:
# Extract annual provincial population for AB and ON from the main dataset.
# Population is recorded per weekly row; we take the mean per (province, year)
# to smooth any mid-year updates.
main_df = pd.read_csv(
    os.path.join(REPO_DIR, "data", "processed", "final_modeling_dataset.csv"),
    parse_dates=["period_start_date"],
    usecols=["iso_3166_2", "period_start_date", "time_scale", "population", "disease"],
)

pop_df = (
    main_df[
        (main_df["time_scale"]   == "wk") &
        (main_df["iso_3166_2"].isin(PROVINCES)) &
        (main_df["disease"]      == "influenza")   # one disease to avoid duplicates
    ]
    .assign(year=lambda d: d["period_start_date"].dt.year)
    .groupby(["iso_3166_2", "year"])["population"]
    .mean()
    .round()
    .astype(int)
    .reset_index()
    .rename(columns={"iso_3166_2": "province"})
)

print("Population sample (AB + ON, 2004-2021):")
print(pop_df[pop_df["year"].between(2004, 2021)].pivot(index="year", columns="province", values="population").to_string())

---
## 4 · Aggregate LSTM → national annual rate

In [ ]:
def national_annual_rate(preds_df, pred_col, pop_df):
    """
    Compute population-weighted national annual rate from weekly provincial predictions.

    For each (disease, year):
      1. Join weekly rates with provincial annual population.
      2. Average weekly rates within year per province.
      3. Weight by province population: rate = Σ(rate_p * pop_p) / Σ(pop_p).

    Returns DataFrame with columns: disease, year, pred_rate_per_100k, actual_rate_per_100k.
    """
    df = preds_df.merge(pop_df, on=["province", "year"], how="left")

    # Annual mean rate per (disease, province, year)
    annual = (
        df.groupby(["disease", "province", "year"])
        .agg(
            pred_rate   =(pred_col,              "mean"),
            actual_rate =("actual_cases_per_100k", "mean"),
            population  =("population",            "first"),
        )
        .reset_index()
    )

    # Population-weighted national rate
    def weighted_mean(grp):
        total_pop   = grp["population"].sum()
        pred_nat    = (grp["pred_rate"]   * grp["population"]).sum() / total_pop
        actual_nat  = (grp["actual_rate"] * grp["population"]).sum() / total_pop
        return pd.Series({"pred_rate_per_100k": pred_nat, "actual_rate_per_100k": actual_nat})

    national = (
        annual.groupby(["disease", "year"])
        .apply(weighted_mean)
        .reset_index()
    )
    return national


lstm_national = national_annual_rate(lstm_preds, "pred_cases_per_100k", pop_df)
lstm_national["model"] = "LSTM"
print(f"LSTM national annual rates: {len(lstm_national)} rows")
print(lstm_national[lstm_national["disease"] == "influenza"].head(5).to_string(index=False))

---
## 5 · Aggregate ARIMA → national annual rate

In [ ]:
arima_national = national_annual_rate(arima_preds, "pred_cases_per_100k", pop_df)
arima_national["model"] = "ARIMA"
print(f"ARIMA national annual rates: {len(arima_national)} rows")
print(arima_national[arima_national["disease"] == "influenza"].head(5).to_string(index=False))

# Save combined national predictions
national_all = pd.concat([lstm_national, arima_national], ignore_index=True)
out_path = os.path.join(TABLES_DIR, "national_annual_predictions.csv")
national_all.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

---
## 6 · Load PHAC reference & merge

In [ ]:
phac = pd.read_csv(os.path.join(REPO_DIR, "data", "processed", "phac-clean.csv"))

# Map PHAC 'pertussis' → 'whooping-cough' to align with model disease names
phac["disease"] = phac["disease"].replace({"pertussis": "whooping-cough"})

# Restrict to test period
phac_test = phac[phac["year"].between(2004, 2021)].copy()

print(f"PHAC test-period records: {len(phac_test)}")
print(phac_test.groupby("disease")["year"].agg(["min", "max", "count"]))

# Merge each model with PHAC
def merge_with_phac(model_df, phac_df):
    return model_df.merge(
        phac_df[["year", "disease", "rate_per_100k", "cases"]]
              .rename(columns={"rate_per_100k": "phac_rate_per_100k", "cases": "phac_cases"}),
        on=["year", "disease"],
        how="inner",
    )

lstm_vs_phac  = merge_with_phac(lstm_national,  phac_test)
arima_vs_phac = merge_with_phac(arima_national, phac_test)

print(f"\nMerged rows — LSTM: {len(lstm_vs_phac)}, ARIMA: {len(arima_vs_phac)}")

---
## 7 · Comparison metrics

In [ ]:
def compute_phac_metrics(df, model_name):
    """Compute MAE, RMSE, and MAPE between model predicted rate and PHAC rate."""
    rows = []
    for disease, grp in df.groupby("disease"):
        error = grp["pred_rate_per_100k"] - grp["phac_rate_per_100k"]
        pct_error = (error / grp["phac_rate_per_100k"].replace(0, np.nan)).abs() * 100
        rows.append({
            "model":              model_name,
            "disease":            disease,
            "n_years":            len(grp),
            "mae":                round(error.abs().mean(), 4),
            "rmse":               round(np.sqrt((error ** 2).mean()), 4),
            "mape":               round(pct_error.mean(), 2),
            "mean_pred_rate":     round(grp["pred_rate_per_100k"].mean(), 4),
            "mean_phac_rate":     round(grp["phac_rate_per_100k"].mean(), 4),
        })
    return pd.DataFrame(rows)


lstm_metrics  = compute_phac_metrics(lstm_vs_phac,  "LSTM")
arima_metrics = compute_phac_metrics(arima_vs_phac, "ARIMA")
metrics_all   = pd.concat([lstm_metrics, arima_metrics], ignore_index=True)

print("PHAC validation metrics (predicted AB+ON rate vs PHAC national rate):")
print(metrics_all.to_string(index=False))

metrics_path = os.path.join(TABLES_DIR, "phac_validation_comparison.csv")
metrics_all.to_csv(metrics_path, index=False)
print(f"\nSaved: {metrics_path}")

---
## 8 · Visualizations

In [ ]:
DISEASE_LABELS = {
    "influenza":     "Influenza",
    "measles":        "Measles",
    "whooping-cough": "Whooping Cough (Pertussis)",
}

fig, axes = plt.subplots(3, 1, figsize=(12, 14))
fig.suptitle(
    "Population-Weighted AB+ON Annual Rate vs PHAC National Rate (2004–2021)",
    fontsize=13, fontweight="bold", y=1.01,
)

for ax, disease in zip(axes, DISEASES):
    phac_row   = phac_test[phac_test["disease"] == disease].sort_values("year")
    lstm_row   = lstm_national[lstm_national["disease"] == disease].sort_values("year")
    arima_row  = arima_national[arima_national["disease"] == disease].sort_values("year")
    actual_row = lstm_national[lstm_national["disease"] == disease].sort_values("year")

    ax.plot(phac_row["year"],    phac_row["rate_per_100k"],        "k-o",  lw=2,   ms=5, label="PHAC (national)",     zorder=5)
    ax.plot(actual_row["year"],  actual_row["actual_rate_per_100k"], "--",  lw=1.5, color="gray", label="Actual (AB+ON weighted)", zorder=4)
    ax.plot(lstm_row["year"],    lstm_row["pred_rate_per_100k"],   "b-s",  lw=1.5, ms=5, label="LSTM predicted",      zorder=3)
    ax.plot(arima_row["year"],   arima_row["pred_rate_per_100k"],  "r-^",  lw=1.5, ms=5, label="ARIMA predicted",     zorder=3)

    ax.set_title(DISEASE_LABELS[disease], fontsize=11)
    ax.set_ylabel("Rate per 100k")
    ax.set_xlabel("Year")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "phac_validation_timeseries.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

In [ ]:
# ── Error scatter: predicted rate vs PHAC rate ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle("Predicted Rate vs PHAC National Rate by Disease", fontsize=12, fontweight="bold")

for ax, disease in zip(axes, DISEASES):
    lstm_d  = lstm_vs_phac[lstm_vs_phac["disease"]  == disease]
    arima_d = arima_vs_phac[arima_vs_phac["disease"] == disease]

    all_vals = pd.concat([
        lstm_d[["phac_rate_per_100k", "pred_rate_per_100k"]],
        arima_d[["phac_rate_per_100k", "pred_rate_per_100k"]],
    ])
    lo = all_vals.min().min() * 0.9
    hi = all_vals.max().max() * 1.1

    ax.scatter(lstm_d["phac_rate_per_100k"],  lstm_d["pred_rate_per_100k"],
               color="blue", alpha=0.7, s=40, label="LSTM",  zorder=4)
    ax.scatter(arima_d["phac_rate_per_100k"], arima_d["pred_rate_per_100k"],
               color="red",  alpha=0.7, s=40, label="ARIMA", zorder=4)
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="Perfect prediction", zorder=3)

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("PHAC rate per 100k")
    ax.set_ylabel("Predicted rate per 100k")
    ax.set_title(DISEASE_LABELS[disease], fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "phac_validation_scatter.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

---
## 9 · Summary

In [ ]:
print("=" * 65)
print("PHAC VALIDATION SUMMARY")
print("=" * 65)
print(f"Scope          : AB + ON (population-weighted average rate)")
print(f"Test period    : 2004–2021")
print(f"Comparison     : Directional sanity check vs PHAC national rate")
print()
for model in ["LSTM", "ARIMA"]:
    m = metrics_all[metrics_all["model"] == model]
    print(f"── {model} ──")
    for _, row in m.iterrows():
        print(f"  {row['disease']:<20}  MAE={row['mae']:.3f}  RMSE={row['rmse']:.3f}  MAPE={row['mape']:.1f}%")
        print(f"    mean predicted: {row['mean_pred_rate']:.3f}  |  mean PHAC: {row['mean_phac_rate']:.3f} per 100k")
    print()
print("Note: MAPE reflects difference between AB+ON weighted rate and")
print("PHAC national rate. Gaps are expected given 2-province scope.")